In [0]:

display(dbutils.fs.ls("/Volumes/workspace/default/retail_data/"))


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/retail_data/customers.csv,customers.csv,18436724,1792094033000
dbfs:/Volumes/workspace/default/retail_data/orders.csv,orders.csv,27089313,1792094036000
dbfs:/Volumes/workspace/default/retail_data/products.csv,products.csv,42566,1792094029000
dbfs:/Volumes/workspace/default/retail_data/regions.csv,regions.csv,667,1792094029000


In [0]:

base_path = "/Volumes/workspace/default/retail_data"

customers_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/customers.csv")
orders_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/orders.csv")
products_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/products.csv")
regions_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/regions.csv")

print("Customers:", customers_df.count())
print("Orders:", orders_df.count())
print("Products:", products_df.count())
print("Regions:", regions_df.count())


Customers: 150000
Orders: 250000
Products: 500
Regions: 16


In [0]:

print("CUSTOMERS")
customers_df.printSchema()

print("ORDERS")
orders_df.printSchema()

print("PRODUCTS")
products_df.printSchema()

print("REGIONS")
regions_df.printSchema()


CUSTOMERS
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)

ORDERS
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: 

In [0]:

display(customers_df.limit(5))
display(orders_df.limit(5))
display(products_df.limit(5))
display(regions_df.limit(5))


customer_id,first_name,last_name,email,phone,gender,date_of_birth,signup_date,city,state,region_id,customer_segment
C0000001,Farhan,Agarwal,customer0000001@example.com,919765005632,Male,1984-09-13,2017-04-25,Gurugram,Haryana,R013,Corporate
C0000002,Sai,Agarwal,customer0000002@mail.test,916254710009,Female,1997-06-28,2017-06-07,Ahmedabad,Gujarat,R005,Corporate
C0000003,Ananya,Nair,customer0000003@sample.org,918089714564,Male,1994-10-07,2024-04-08,Chennai,Tamil Nadu,R007,Consumer
C0000004,Aditya,Singh,customer0000004@demo.net,919605880228,Male,1958-04-19,2021-05-01,Patna,Bihar,R010,Consumer
C0000005,Kavya,Mehta,customer0000005@example.com,919202038742,Male,1998-09-02,2023-09-16,Indore,Madhya Pradesh,R011,Consumer


order_id,customer_id,product_id,order_date,quantity,unit_price,discount_pct,gross_amount,discount_amount,net_amount,order_status,payment_method,shipping_days,delivery_date,region_id
O00000001,C0145417,P0388,2023-05-13,11,331.13,0.2,3642.43,728.49,2913.94,Shipped,Debit Card,6,NaT,R013
O00000002,C0082464,P0383,2021-01-20,6,1781.29,0.03,10687.74,320.63,10367.11,Delivered,Cash on Delivery,2,2021-01-22,R010
O00000003,C0086403,P0017,2021-01-02,12,598.84,0.25,7186.08,1796.52,5389.56,Delivered,UPI,4,2021-01-06,R002
O00000004,C0005411,P0149,2026-07-28,4,345.49,0.03,1381.96,41.46,1340.5,Delivered,UPI,10,2026-08-07,R009
O00000005,C0059216,P0337,2022-06-17,11,552.27,0.1,6074.97,607.5,5467.47,Shipped,UPI,4,NaT,R003


product_id,product_name,category,brand,unit_price,unit_cost,supplier,stock_quantity,is_active
P0001,UrbanLeaf Router,Electronics,UrbanLeaf,2219.04,1497.98,Supplier 056,1206,true
P0002,Orion Toaster,Home & Kitchen,Orion,66.01,51.97,Supplier 041,2226,true
P0003,Zenith Backpack,Fashion,Zenith,1544.23,727.2,Supplier 143,241,true
P0004,Nova Body Lotion,Beauty & Personal Care,Nova,276.35,217.16,Supplier 147,1224,true
P0005,FreshField Cycling Helmet,Sports & Outdoors,FreshField,111.8,61.15,Supplier 010,1844,true


region_id,region_name,country,state,city,currency
R001,North,India,Delhi,New Delhi,INR
R002,North,India,Uttar Pradesh,Lucknow,INR
R003,North,India,Rajasthan,Jaipur,INR
R004,West,India,Maharashtra,Mumbai,INR
R005,West,India,Gujarat,Ahmedabad,INR


In [0]:

# STEP 1: BRONZE LAYER

catalog_schema = "workspace.default"

source_tables = {
    "bronze_customers": customers_df,
    "bronze_orders": orders_df,
    "bronze_products": products_df,
    "bronze_regions": regions_df
}

for table_name, df in source_tables.items():
    full_table_name = f"{catalog_schema}.{table_name}"

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_table_name)

    print(f"Created: {full_table_name}")

print("Bronze layer completed successfully!")


Created: workspace.default.bronze_customers
Created: workspace.default.bronze_orders
Created: workspace.default.bronze_products
Created: workspace.default.bronze_regions
Bronze layer completed successfully!


In [0]:
%sql

SHOW TABLES IN workspace.default;


database,tableName,isTemporary
default,bronze_customers,false
default,bronze_orders,false
default,bronze_products,false
default,bronze_regions,false


In [0]:
%sql


SELECT 'customers' AS table_name, COUNT(*) AS total_rows
FROM workspace.default.bronze_customers

UNION ALL

SELECT 'orders', COUNT(*)
FROM workspace.default.bronze_orders

UNION ALL

SELECT 'products', COUNT(*)
FROM workspace.default.bronze_products

UNION ALL

SELECT 'regions', COUNT(*)
FROM workspace.default.bronze_regions;


table_name,total_rows
customers,150000
orders,250000
products,500
regions,16


In [0]:

from pyspark.sql import functions as F

# Read data from Bronze tables
customers_bronze = spark.table("workspace.default.bronze_customers")
orders_bronze = spark.table("workspace.default.bronze_orders")
products_bronze = spark.table("workspace.default.bronze_products")
regions_bronze = spark.table("workspace.default.bronze_regions")

# Check missing IDs and duplicate IDs
def quality_check(df, table_name, id_column):
    result = df.agg(
        F.count("*").alias("total_rows"),
        F.sum(
            F.when(F.col(id_column).isNull(), 1).otherwise(0)
        ).alias("null_ids"),
        F.countDistinct(id_column).alias("unique_ids")
    ).first()

    print(f"\n{table_name}")
    print(f"Total rows: {result['total_rows']}")
    print(f"Null IDs: {result['null_ids'] or 0}")
    print(f"Duplicate IDs: {result['total_rows'] - (result['unique_ids'] or 0) - (result['null_ids'] or 0)}")

quality_check(customers_bronze, "Customers", "customer_id")
quality_check(orders_bronze, "Orders", "order_id")
quality_check(products_bronze, "Products", "product_id")
quality_check(regions_bronze, "Regions", "region_id")



Customers
Total rows: 150000
Null IDs: 0
Duplicate IDs: 0

Orders
Total rows: 250000
Null IDs: 0
Duplicate IDs: 0

Products
Total rows: 500
Null IDs: 0
Duplicate IDs: 0

Regions
Total rows: 16
Null IDs: 0
Duplicate IDs: 0


In [0]:

# Remove rows with missing IDs and duplicate IDs
customers_silver = (
    customers_bronze
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

orders_silver = (
    orders_bronze
    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
)

products_silver = (
    products_bronze
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
)

regions_silver = (
    regions_bronze
    .filter(F.col("region_id").isNotNull())
    .dropDuplicates(["region_id"])
)

# Save Silver Delta tables
silver_tables = {
    "silver_customers": customers_silver,
    "silver_orders": orders_silver,
    "silver_products": products_silver,
    "silver_regions": regions_silver
}

for table_name, df in silver_tables.items():
    df.write.format("delta").mode("overwrite").saveAsTable(
        f"workspace.default.{table_name}"
    )
    print(f"Created workspace.default.{table_name}")

print("Silver layer created successfully!")


Created workspace.default.silver_customers
Created workspace.default.silver_orders
Created workspace.default.silver_products
Created workspace.default.silver_regions
Silver layer created successfully!


In [0]:
%sql


SELECT 'silver_customers' AS table_name, COUNT(*) AS total_rows
FROM workspace.default.silver_customers

UNION ALL

SELECT 'silver_orders', COUNT(*)
FROM workspace.default.silver_orders

UNION ALL

SELECT 'silver_products', COUNT(*)
FROM workspace.default.silver_products

UNION ALL

SELECT 'silver_regions', COUNT(*)
FROM workspace.default.silver_regions;


table_name,total_rows
silver_customers,150000
silver_orders,250000
silver_products,500
silver_regions,16


In [0]:

from pyspark.sql import functions as F

# Read Silver tables
customers = spark.table("workspace.default.silver_customers")
orders = spark.table("workspace.default.silver_orders")
products = spark.table("workspace.default.silver_products")
regions = spark.table("workspace.default.silver_regions")

# Join the datasets
gold_sales = (
    orders.alias("o")
    .join(customers.alias("c"), "customer_id", "left")
    .join(products.alias("p"), "product_id", "left")
    .join(
        regions.select(
            "region_id",
            F.col("region_name"),
            F.col("country").alias("region_country")
        ).alias("r"),
        orders["region_id"] == F.col("r.region_id"),
        "left"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.order_date"),
        F.col("o.customer_id"),
        F.concat_ws(
            " ",
            F.col("c.first_name"),
            F.col("c.last_name")
        ).alias("customer_name"),
        F.col("c.customer_segment"),
        F.col("o.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("o.quantity"),
        F.col("o.unit_price").alias("order_unit_price"),
        F.col("o.gross_amount"),
        F.col("o.discount_amount"),
        F.col("o.net_amount"),
        F.col("o.order_status"),
        F.col("o.payment_method"),
        F.col("o.shipping_days"),
        F.col("o.region_id"),
        F.col("r.region_name"),
        F.col("r.region_country")
    )
)

# Save Gold table
gold_sales.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_sales"
)

print("Gold sales table created!")
print("Gold sales rows:", gold_sales.count())


Gold sales table created!
Gold sales rows: 250000


In [0]:

gold_sales = spark.table("workspace.default.gold_sales")

monthly_sales = (
    gold_sales
    .filter(F.col("order_date").isNotNull())
    .groupBy(F.date_format("order_date", "yyyy-MM").alias("sales_month"))
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_units_sold"),
        F.round(F.sum("gross_amount"), 2).alias("gross_revenue"),
        F.round(F.sum("discount_amount"), 2).alias("total_discount"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue")
    )
    .orderBy("sales_month")
)

monthly_sales.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_monthly_sales"
)

display(monthly_sales)


sales_month,total_orders,total_units_sold,gross_revenue,total_discount,net_revenue
2021-01,3587,23082,1.9502253353E8,1.677818105E7,1.7824435248E8
2021-02,3327,21670,1.8399786122E8,1.601727052E7,1.679805907E8
2021-03,3728,23919,2.1068637413E8,1.85164139E7,1.9216996023E8
2021-04,3529,22951,1.9210986899E8,1.716281517E7,1.7494705382E8
2021-05,3591,23122,2.068313444E8,1.744620221E7,1.8938514219E8
2021-06,3546,23067,1.9554602968E8,1.69182343E7,1.7862779538E8
2021-07,3632,23775,2.0744499066E8,1.80193887E7,1.8942560196E8
2021-08,3739,24079,2.1339370069E8,1.94361305E7,1.9395757019E8
2021-09,3454,22305,1.895234377E8,1.65555201E7,1.729679176E8
2021-10,3640,23365,2.0085777508E8,1.67390516E7,1.8411872348E8


In [0]:

product_performance = (
    gold_sales
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue")
    )
    .orderBy(F.desc("net_revenue"))
)

product_performance.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_product_performance"
)

display(product_performance.limit(20))


product_id,product_name,category,total_orders,units_sold,net_revenue
P0155,BluePeak Speaker,Electronics,517,3376,3.1026038835E8
P0414,FreshField Mouse,Electronics,525,3383,3.0018034244E8
P0113,Orion Headphones,Electronics,524,3395,2.9245277897E8
P0190,Aster Tablet,Electronics,501,3222,2.8751663729E8
P0120,UrbanLeaf Tablet,Electronics,497,3164,2.8610482636E8
P0050,Everwell Keyboard,Electronics,468,3084,2.7915931308E8
P0218,Orion Keyboard,Electronics,542,3593,2.726500899E8
P0169,DailyChoice Laptop,Electronics,525,3394,2.2812735699E8
P0372,BluePeak Keyboard,Electronics,544,3627,2.0910938366E8
P0484,Aster Router,Electronics,512,3500,1.8636917166E8


In [0]:
%sql

SELECT
    region_name,
    category,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS units_sold,
    ROUND(SUM(net_amount), 2) AS net_revenue
FROM workspace.default.gold_sales
GROUP BY region_name, category
ORDER BY net_revenue DESC;


region_name,category,total_orders,units_sold,net_revenue
North,Electronics,11236,73495,1.49108561044E9
South,Electronics,9017,58613,1.17977160681E9
West,Electronics,6841,44203,8.7537314618E8
Central,Electronics,4666,29873,6.0184954574E8
East,Electronics,4426,29055,5.8960330025E8
North,Sports & Outdoors,11125,71713,4.8070769641E8
North,Grocery,11077,71752,4.7262121352E8
North,Office Supplies,10941,70783,4.4973425766E8
North,Beauty & Personal Care,11077,71255,4.4382043486E8
North,Home & Kitchen,11163,73009,4.3772874585E8


In [0]:

from pyspark.sql import functions as F

gold_sales = spark.table("workspace.default.gold_sales")

customer_performance = (
    gold_sales
    .groupBy(
        "customer_id",
        "customer_name",
        "customer_segment",
        "region_name"
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_units_purchased"),
        F.round(F.sum("net_amount"), 2).alias("total_spent"),
        F.max("order_date").alias("last_order_date")
    )
    .orderBy(F.desc("total_spent"))
)

customer_performance.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_customer_performance"
)

display(customer_performance.limit(20))


customer_id,customer_name,customer_segment,region_name,total_orders,total_units_purchased,total_spent,last_order_date
C0050347,Meera Verma,Consumer,East,3,31,2483567.37,2022-07-26
C0054439,Rohan Thakur,Consumer,South,3,32,2117317.88,2026-10-10
C0087369,Sai Choudhary,Small Business,West,2,23,1887551.93,2026-06-22
C0087653,Tanya Kulkarni,Corporate,North,6,46,1837413.17,2026-09-18
C0125939,Kabir Singh,Consumer,North,4,32,1827154.4,2024-11-26
C0023878,Myra Verma,Corporate,South,3,29,1805730.9,2023-11-07
C0086027,Ishita Desai,Small Business,West,7,66,1777561.65,2025-10-03
C0115780,Pooja Choudhary,Consumer,Central,5,40,1694548.23,2026-05-31
C0055348,Diya Patel,Consumer,North,4,40,1689214.14,2023-07-31
C0093263,Riya Singh,Small Business,North,4,39,1683152.81,2025-08-10


In [0]:
%sql

SELECT
    product_name,
    category,
    SUM(quantity) AS total_units_sold,
    ROUND(SUM(net_amount), 2) AS net_revenue
FROM workspace.default.gold_sales
GROUP BY product_name, category
ORDER BY net_revenue DESC
LIMIT 10;


product_name,category,total_units_sold,net_revenue
BluePeak Speaker,Electronics,3376,3.1026038835E8
FreshField Mouse,Electronics,3383,3.0018034244E8
Everwell Keyboard,Electronics,9595,2.9334746765E8
Orion Headphones,Electronics,3395,2.9245277897E8
Aster Tablet,Electronics,3222,2.8751663729E8
UrbanLeaf Tablet,Electronics,3164,2.8610482636E8
Orion Keyboard,Electronics,3593,2.726500899E8
FreshField Smartwatch,Electronics,6405,2.6738876122E8
DailyChoice Laptop,Electronics,3394,2.2812735699E8
BluePeak Keyboard,Electronics,3627,2.0910938366E8


In [0]:
%sql

SELECT
    region_name,
    region_country,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(net_amount), 2) AS net_revenue
FROM workspace.default.gold_sales
GROUP BY region_name, region_country
ORDER BY net_revenue DESC;


region_name,region_country,total_orders,net_revenue
North,India,77724,4.02069791705E9
South,India,62315,3.20912224169E9
West,India,47230,2.39760378907E9
Central,India,31921,1.62791630569E9
East,India,30810,1.57379113717E9


In [0]:

from pyspark.sql import functions as F

orders = spark.table("workspace.default.silver_orders")
products = spark.table("workspace.default.silver_products")

profit_analysis = (
    orders.alias("o")
    .join(
        products.select("product_id", "product_name", "category", "unit_cost").alias("p"),
        "product_id",
        "left"
    )
    .withColumn(
        "estimated_cost",
        F.col("quantity") * F.col("p.unit_cost")
    )
    .withColumn(
        "estimated_profit",
        F.col("net_amount") - F.col("estimated_cost")
    )
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("net_amount"), 2).alias("net_revenue"),
        F.round(F.sum("estimated_cost"), 2).alias("estimated_cost"),
        F.round(F.sum("estimated_profit"), 2).alias("estimated_profit")
    )
    .orderBy(F.desc("estimated_profit"))
)

profit_analysis.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_profit_analysis"
)

display(profit_analysis.limit(20))


product_id,product_name,category,units_sold,net_revenue,estimated_cost,estimated_profit
P0414,FreshField Mouse,Electronics,3383,3.0018034244E8,1.4184516423E8,1.5833517821E8
P0050,Everwell Keyboard,Electronics,3084,2.7915931308E8,1.3938144168E8,1.397778714E8
P0113,Orion Headphones,Electronics,3395,2.9245277897E8,1.5391636505E8,1.3853641392E8
P0190,Aster Tablet,Electronics,3222,2.8751663729E8,1.6381079748E8,1.2370583981E8
P0484,Aster Router,Electronics,3500,1.8636917166E8,9.3134475E7,9.323469666E7
P0155,BluePeak Speaker,Electronics,3376,3.1026038835E8,2.2795059216E8,8.230979619E7
P0218,Orion Keyboard,Electronics,3593,2.726500899E8,1.9767126638E8,7.497882352E7
P0120,UrbanLeaf Tablet,Electronics,3164,2.8610482636E8,2.1302901928E8,7.307580708E7
P0092,Zenith Mouse,Electronics,3199,1.3891553568E8,6.749685264E7,7.141868304E7
P0315,Aster Oats 1kg,Grocery,3429,1.3980618718E8,6.898691943E7,7.081926775E7


In [0]:
%sql

SELECT
    COUNT(*) AS total_orders,
    COUNT_IF(order_id IS NULL) AS missing_order_ids,
    COUNT_IF(order_date IS NULL) AS missing_order_dates,
    COUNT_IF(quantity IS NULL OR quantity <= 0) AS invalid_quantities,
    COUNT_IF(net_amount IS NULL OR net_amount < 0) AS invalid_net_amounts
FROM workspace.default.gold_sales;


total_orders,missing_order_ids,missing_order_dates,invalid_quantities,invalid_net_amounts
250000,0,0,0,0


In [0]:
%sql
SELECT SUM(net_amount) AS total_revenue
FROM gold_sales;

total_revenue
1.282913139067021E10


In [0]:
%sql
SELECT SUM(estimated_profit) AS total_estimated_profit
FROM gold_profit_analysis;

total_estimated_profit
4.1404025317499995E9


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(net_amount) AS total_revenue
FROM workspace.default.gold_sales;

total_rows,total_revenue
250000,1.282913139067021E10


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(net_amount) AS total_revenue,
    MIN(net_amount) AS min_net_amount,
    MAX(net_amount) AS max_net_amount
FROM workspace.default.gold_sales;

total_rows,total_revenue,min_net_amount,max_net_amount
250000,1.282913139067021E10,46.19,1208849.88
